# <font color='white'>1. Read Libraries<font>

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ============================================================================
# ADIM 1: ÇOK BASİT FEATURE ENGİNEERİNG (LAG YOK)
# ============================================================================


def prepare_basic_features(df_):
    """
    Sadece o haftanın bilgilerini kullan, geçmiş bilgi yok
    """
    df = df_.copy()
    
    # Datetime'a çevir
    df['week_start'] = pd.to_datetime(df['week_start'])
    df['customer_created_at'] = pd.to_datetime(df['customer_created_at'])
    
    # === ZAMAN FEATURELARI ===
    df['week'] = df['week_start'].dt.isocalendar().week.astype(int)
    df['month'] = df['week_start'].dt.month.astype(int)
    df['year'] = df['week_start'].dt.year.astype(int)
    df['day_of_year'] = df['week_start'].dt.dayofyear.astype(int)
    
    # === MÜŞTERİ FEATURELARI ===
    # Müşteri kaç gündür platformda?
    df['customer_age_days'] = (df['week_start'] - df['customer_created_at']).dt.days.clip(lower=0).fillna(0).astype(int)

    return df
    

In [3]:
# ============================================================================
# ADIM 2: TRAIN-VALIDATION SPLIT (ÇOK ÖNEMLİ!)
# ============================================================================

def split_for_time_series(df, val_weeks=4):
    """
    Zaman serisine uygun split
    
    Train setinde:
    - week_start: O haftanın tarihi
    - Target_purchase_next_1w: BİR SONRAKİ hafta alınacak mı?
    
    Örnek:
    week_start: 2025-09-08
    Target_purchase_next_1w: 0  → 2025-09-15'te alınmadı
    
    Yani train seti 2025-09-08'de bitiyor ama aslında
    2025-09-15'e kadarki bilgiyi içeriyor (target olarak)
    """
    
    df = df.copy()
    
    # Sadece Target'ı olan satırları kullan (eğer NaN varsa)
    df_with_target = df[df['Target_purchase_next_1w'].notna()].copy()
    
    unique_weeks = np.sort(df_with_target['week_start'].unique())
    
    print("="*70)
    print("TRAIN-VALIDATION SPLIT")
    print("="*70)
    print(f"Toplam hafta sayısı (week_start): {len(unique_weeks)}")
    print(f"İlk hafta (week_start): {unique_weeks[0]}")
    print(f"Son hafta (week_start): {unique_weeks[-1]}")
    
    # # Son hafta için target varsa, o hafta + 1 hafta bilgisi var demektir
    # # Örnek: week_start=2025-09-08 varsa, 2025-09-15'teki bilgi de var (target olarak)
    # last_week_as_date = pd.to_datetime(unique_weeks[-1])
    # actual_coverage = last_week_as_date + pd.Timedelta(days=7)
    # print(f"Gerçek kapsama (target dahil): {actual_coverage.date()}")
    
    # Son val_weeks hafta validation olsun
    val_cutoff = unique_weeks[-val_weeks] # Biz 4 hafta dedik.
    
    print(f"\nValidation başlangıcı: {val_cutoff}")
    
    # Split
    train_mask = df_with_target['week_start'] < val_cutoff
    val_mask = df_with_target['week_start'] >= val_cutoff
    
    df_train = df_with_target[train_mask].copy()
    df_val = df_with_target[val_mask].copy()
    
    print(f"\nTrain:")
    print(f"  - Hafta sayısı: {len(df_train['week_start'].unique())}")
    print(f"  - İlk hafta: {df_train['week_start'].min()}")
    print(f"  - Son hafta: {df_train['week_start'].max()}")
    print(f"  - Satır sayısı: {len(df_train):,}")
    
    print(f"\nValidation:")
    print(f"  - Hafta sayısı: {len(df_val['week_start'].unique())}")
    print(f"  - İlk hafta: {df_val['week_start'].min()}")
    print(f"  - Son hafta: {df_val['week_start'].max()}")
    print(f"  - Satır sayısı: {len(df_val):,}")
    
    return df_train, df_val

In [ ]:
deneme2

In [4]:
# ============================================================================
# ADIM 3: KATEGORİK DEĞİŞKENLERİ ENCODE ET
# ============================================================================

def encode_categorical_features(df_train, df_val, df_test=None):
    """
    Kategorik değişkenleri encode et:
    - Düşük kardinalite (az kategori): One-Hot Encoding
    - Yüksek kardinalite (çok kategori): Label Encoding
    """
    
    # Düşük kardinaliteli: One-Hot Encoding
    # (customer_category, customer_status, grade_name, unit_name)
    low_cardinality_cols = ['customer_category', 'customer_status', 'grade_name', 'unit_name']
    low_cardinality_cols = [col for col in low_cardinality_cols if col in df_train.columns]
    
    # Yüksek kardinaliteli: Label Encoding  
    # (customer_id, product_id)
    high_cardinality_cols = ['customer_id', 'product_id']
    high_cardinality_cols = [col for col in high_cardinality_cols if col in df_train.columns]
    
    print("\n--- ENCODING STRATEJİSİ ---")
    print(f"One-Hot Encoding: {low_cardinality_cols}")
    print(f"Label Encoding: {high_cardinality_cols}")
    
    # ========================================================================
    # ONE-HOT ENCODING (Düşük Kardinalite)
    # ========================================================================
    
    if low_cardinality_cols:
        print("\n--- ONE-HOT ENCODING ---")
        
        for col in low_cardinality_cols:
            # Train'deki benzersiz değerleri öğren
            unique_values = df_train[col].astype(str).unique()
            print(f"\n{col}: {len(unique_values)} benzersiz değer")
            print(f"  Değerler: {list(unique_values)[:5]}...")
            
            # One-hot encoding (train'den öğren)
            for val in unique_values:
                col_name = f"{col}_{val}"
                df_train[col_name] = (df_train[col].astype(str) == val).astype(int)
                df_val[col_name] = (df_val[col].astype(str) == val).astype(int)
                
                if df_test is not None:
                    df_test[col_name] = (df_test[col].astype(str) == val).astype(int)
        
        # Orijinal sütunları çıkar (isteğe bağlı)
        # df_train = df_train.drop(columns=low_cardinality_cols)
        # df_val = df_val.drop(columns=low_cardinality_cols)
        # if df_test is not None:
        #     df_test = df_test.drop(columns=low_cardinality_cols)
    
    # ========================================================================
    # LABEL ENCODING (Yüksek Kardinalite)
    # ========================================================================
    
    encoders = {}
    
    if high_cardinality_cols:
        print("\n--- LABEL ENCODING ---")
        
        for col in high_cardinality_cols:
            # Encoder oluştur
            encoder = LabelEncoder()
            
            # Train'deki tüm değerleri öğren
            encoder.fit(df_train[col].astype(str))
            
            print(f"\n{col}: {len(encoder.classes_)} benzersiz değer")
            
            # Train'i encode et
            df_train[f'{col}_encoded'] = encoder.transform(df_train[col].astype(str))
            
            # Validation'ı encode et (yeni değerler için -1)
            df_val[f'{col}_encoded'] = df_val[col].astype(str).map(
                lambda x: encoder.transform([x])[0] if x in encoder.classes_ else -1
            )
            
            # Test varsa
            if df_test is not None:
                df_test[f'{col}_encoded'] = df_test[col].astype(str).map(
                    lambda x: encoder.transform([x])[0] if x in encoder.classes_ else -1
                )
            
            encoders[col] = encoder
    
    return df_train, df_val, df_test, encoders

In [ ]:
# ============================================================================
# ADIM 4: MODEL EĞİTİMİ
# ============================================================================

def train_simple_model(df_train, df_val,threshold):
    """
    Basit Logistic Regression
    """
    
    # === FEATURE SEÇİMİ ===
    # DİKKAT: Sadece test'te de bilinebilecek feature'ları kullan!
    numeric_features = [
        # Zaman (test'te bilinir)
        'week', 'month', 'year', 'day_of_year',
        
        # Müşteri (test'te bilinir)
        'customer_age_days',
        
        # Ürün (test'te bilinir - katalogda var)
        # 'selling_price',
        
        # ÖNEMLİ: Aşağıdakileri KULLANMA!
        # 'purchased_this_week',  # ← Test'te bilinmez!
        # 'qty_this_week',        # ← Test'te bilinmez!
        # 'spend_this_week',      # ← Test'te bilinmez!
        # 'num_orders_week',      # ← Test'te bilinmez!
    ]
    
    # One-hot encoded sütunları ekle
    onehot_cols = [col for col in df_train.columns if any(
        col.startswith(f'{cat}_') for cat in ['customer_category', 'customer_status', 'grade_name', 'unit_name']
    )]
    
    # Label encoded sütunları ekle
    label_encoded_cols = [col for col in df_train.columns if col.endswith('_encoded')]
    
    features = numeric_features + onehot_cols + label_encoded_cols
    
    # Sadece mevcut featureları kullan
    features = [f for f in features if f in df_train.columns]
    
    print("\n" + "="*70)
    print("MODEL EĞİTİMİ")
    print("="*70)
    print(f"Kullanılan feature sayısı: {len(features)}")
    print(f"\nFeature grupları:")
    print(f"  - Numeric: {len(numeric_features)}")
    print(f"  - One-Hot Encoded: {len(onehot_cols)}")
    print(f"  - Label Encoded: {len(label_encoded_cols)}")
    
    if len(onehot_cols) > 0:
        print(f"\n  One-Hot örnek sütunlar: {onehot_cols[:5]}...")
    if len(label_encoded_cols) > 0:
        print(f"  Label Encoded sütunlar: {label_encoded_cols}")
    
    # === VERİ HAZIRLAMA ===
    X_train = df_train[features].fillna(0)
    y_train = df_train['Target_purchase_next_1w']
    
    X_val = df_val[features].fillna(0)
    y_val = df_val['Target_purchase_next_1w']
    
    # Target dağılımı
    print(f"\nTarget Dağılımı:")
    print(f"Train - Satın alacak: {y_train.sum():,} / {len(y_train):,} ({y_train.mean()*100:.2f}%)")
    print(f"Val   - Satın alacak: {y_val.sum():,} / {len(y_val):,} ({y_val.mean()*100:.2f}%)")
    
    # === MODEL ===
    model = LogisticRegression(
        max_iter=1000,
        random_state=42,
        class_weight='balanced',  # Dengesiz veri için ?
        C=0.1  # Regularization (küçültürsen daha basit model)
    )
    
    print(f"\nModel eğitiliyor...")
    model.fit(X_train, y_train)
    
    # === TAHMİNLER ===
    y_train_pred_proba = model.predict_proba(X_train)[:, 1]
    y_val_pred_proba = model.predict_proba(X_val)[:, 1]
    
    y_train_pred = (y_train_pred_proba > threshold).astype(int)
    y_val_pred = (y_val_pred_proba > threshold).astype(int)
    
    # === METRIKLER ===
    train_auc = roc_auc_score(y_train, y_train_pred_proba)
    val_auc = roc_auc_score(y_val, y_val_pred_proba)
    
    print("\n" + "="*70)
    print("SONUÇLAR")
    print("="*70)
    print(f"Train AUC: {train_auc:.4f}")
    print(f"Val AUC:   {val_auc:.4f}")
    print(f"Fark:      {abs(train_auc - val_auc):.4f}")
    
    if abs(train_auc - val_auc) > 0.05:
        print("  ⚠️  Overfitting olabilir!")
    elif abs(train_auc - val_auc) < 0.01:
        print("  ✓  Model stabil görünüyor")
    
    # Confusion Matrix
    print(f"\nValidation Confusion Matrix:")
    cm = confusion_matrix(y_val, y_val_pred)
    print(cm)
    print(f"  - True Negatives:  {cm[0,0]:,}")
    print(f"  - False Positives: {cm[0,1]:,}")
    print(f"  - False Negatives: {cm[1,0]:,}")
    print(f"  - True Positives:  {cm[1,1]:,}")
    
    # Feature Importance (sadece numeric ve label encoded için)
    print(f"\n" + "="*70)
    print("EN ÖNEMLİ FEATURE'LAR (Mutlak Katsayı)")
    print("="*70)
    
    feature_importance = pd.DataFrame({
        'feature': features,
        'coefficient': model.coef_[0]
    })
    feature_importance['abs_coef'] = feature_importance['coefficient'].abs()
    feature_importance = feature_importance.sort_values('abs_coef', ascending=False)
    
    print(feature_importance[['feature', 'coefficient']].head(15).to_string(index=False))
    
    return model, features

In [15]:
# ============================================================================
# ADIM 5: TEST SETİ İÇİN TAHMİN
# ============================================================================

def predict_test(df_test, model, features,threshold):
    """
    Test seti için tahmin yap
    """
    
    print("\n" + "="*70)
    print("TEST TAHMİNİ")
    print("="*70)
    
    # Test setini hazırla
    X_test = df_test[features].fillna(0)
    
    print(f"Test satır sayısı: {len(X_test):,}")
    print(f"Test haftası: {df_test['week_start'].unique()}")
    
    # Eksik feature'ları kontrol et
    missing_features = [f for f in features if f not in df_test.columns]
    if missing_features:
        print(f"\n⚠️  Uyarı: Test setinde eksik feature'lar var:")
        for f in missing_features:
            print(f"  - {f}")
        print("\nBu feature'lar 0 ile dolduruldu.")
    
    # Tahmin
    predictions_proba = model.predict_proba(X_test)[:, 1]
    predictions_binary = (predictions_proba > threshold).astype(int)
    
    # Sonuç
    df_result = df_test.copy()
    df_result['prediction_proba'] = predictions_proba
    df_result['prediction_binary'] = predictions_binary
    
    print(f"\nTahmin Özeti:")
    print(f"  - Satın alacak tahmin edilen: {predictions_binary.sum():,} / {len(predictions_binary):,} ({predictions_binary.mean()*100:.2f}%)")
    print(f"  - Ortalama olasılık: {predictions_proba.mean():.4f}")
    print(f"  - Min olasılık: {predictions_proba.min():.4f}")
    print(f"  - Max olasılık: {predictions_proba.max():.4f}")
    
    return df_result

In [8]:
df_train_raw = pd.read_csv("../data/train.csv")

df_test_raw = pd.read_csv("../data/test.csv")

In [9]:
# Test setini filtrele (sadece 2025-09-22)
df_test_sample = df_test_raw[df_test_raw['week_start'] == '2025-09-22'].copy()

In [10]:
"""
Tüm süreci çalıştır
"""

print("="*70)
print("BASİT ZAMAN SERİSİ TAHMİN PİPELİNE")
print("="*70)

# 1. Feature Engineering
print("\n[1/6] Feature Engineering...")
df_train = prepare_basic_features(df_train_raw)
df_test = prepare_basic_features(df_test_sample)

# 2. Train-Validation Split
print("\n[2/6] Train-Validation Split...")
df_train_split, df_val_split = split_for_time_series(df_train, val_weeks=4)


# 3. Kategorik Encode
print("\n[3/6] Kategorik değişkenler encode ediliyor...")
df_train_split, df_val_split, df_test, encoders = encode_categorical_features(
    df_train_split, df_val_split, df_test
)


BASİT ZAMAN SERİSİ TAHMİN PİPELİNE

[1/6] Feature Engineering...

[2/6] Train-Validation Split...
TRAIN-VALIDATION SPLIT
Toplam hafta sayısı (week_start): 46
İlk hafta (week_start): 2024-10-28T00:00:00.000000000
Son hafta (week_start): 2025-09-08T00:00:00.000000000

Validation başlangıcı: 2025-08-18T00:00:00.000000000

Train:
  - Hafta sayısı: 42
  - İlk hafta: 2024-10-28 00:00:00
  - Son hafta: 2025-08-11 00:00:00
  - Satır sayısı: 1,930,572

Validation:
  - Hafta sayısı: 4
  - İlk hafta: 2025-08-18 00:00:00
  - Son hafta: 2025-09-08 00:00:00
  - Satır sayısı: 183,864

[3/6] Kategorik değişkenler encode ediliyor...

--- ENCODING STRATEJİSİ ---
One-Hot Encoding: ['customer_category', 'customer_status', 'grade_name', 'unit_name']
Label Encoding: ['customer_id', 'product_id']

--- ONE-HOT ENCODING ---

customer_category: 8 benzersiz değer
  Değerler: ['CUST_CAT_003', 'CUST_CAT_000', 'CUST_CAT_006', 'CUST_CAT_005', 'CUST_CAT_002']...

customer_status: 4 benzersiz değer
  Değerler: ['CUST_

In [ ]:

# 4. Model Eğitimi
print("\n[4/6] Model eğitimi...")
model, features = train_simple_model(df_train_split, df_val_split,0.5)

# 5. Test Tahmini
print("\n[5/6] Test tahmini...")
df_test_results = predict_test(df_test, model, features,0.5)


In [13]:
# 4. Model Eğitimi
print("\n[4/6] Model eğitimi...")
model, features = train_simple_model(df_train_split, df_val_split,0.7)

# 5. Test Tahmini
print("\n[5/6] Test tahmini...")
df_test_results = predict_test(df_test, model, features,0.7)


[4/6] Model eğitimi...

MODEL EĞİTİMİ
Kullanılan feature sayısı: 37

Feature grupları:
  - Numeric: 5
  - One-Hot Encoded: 30
  - Label Encoded: 2

  One-Hot örnek sütunlar: ['customer_category_CUST_CAT_003', 'customer_category_CUST_CAT_000', 'customer_category_CUST_CAT_006', 'customer_category_CUST_CAT_005', 'customer_category_CUST_CAT_002']...
  Label Encoded sütunlar: ['customer_id_encoded', 'product_id_encoded']

Target Dağılımı:
Train - Satın alacak: 37,797 / 1,930,572 (1.96%)
Val   - Satın alacak: 5,771 / 183,864 (3.14%)

Model eğitiliyor...

SONUÇLAR
Train AUC: 0.8260
Val AUC:   0.7273
Fark:      0.0987
  ⚠️  Overfitting olabilir!

Validation Confusion Matrix:
[[158051  20042]
 [  4037   1734]]
  - True Negatives:  158,051
  - False Positives: 20,042
  - False Negatives: 4,037
  - True Positives:  1,734

EN ÖNEMLİ FEATURE'LAR (Mutlak Katsayı)
                       feature  coefficient
           grade_name_GRADE_05    -2.160832
customer_category_CUST_CAT_003     1.489344
custo

In [16]:
# C parametresini küçülttüm. 

# 4. Model Eğitimi
print("\n[4/6] Model eğitimi...")
model, features = train_simple_model(df_train_split, df_val_split,0.7)

# 5. Test Tahmini
print("\n[5/6] Test tahmini...")
df_test_results = predict_test(df_test, model, features,0.7)


[4/6] Model eğitimi...

MODEL EĞİTİMİ
Kullanılan feature sayısı: 37

Feature grupları:
  - Numeric: 5
  - One-Hot Encoded: 30
  - Label Encoded: 2

  One-Hot örnek sütunlar: ['customer_category_CUST_CAT_003', 'customer_category_CUST_CAT_000', 'customer_category_CUST_CAT_006', 'customer_category_CUST_CAT_005', 'customer_category_CUST_CAT_002']...
  Label Encoded sütunlar: ['customer_id_encoded', 'product_id_encoded']

Target Dağılımı:
Train - Satın alacak: 37,797 / 1,930,572 (1.96%)
Val   - Satın alacak: 5,771 / 183,864 (3.14%)

Model eğitiliyor...

SONUÇLAR
Train AUC: 0.8276
Val AUC:   0.7334
Fark:      0.0942
  ⚠️  Overfitting olabilir!

Validation Confusion Matrix:
[[156597  21496]
 [  3975   1796]]
  - True Negatives:  156,597
  - False Positives: 21,496
  - False Negatives: 3,975
  - True Positives:  1,796

EN ÖNEMLİ FEATURE'LAR (Mutlak Katsayı)
                       feature  coefficient
           grade_name_GRADE_05    -2.510489
customer_category_CUST_CAT_001    -1.759346
custo

In [11]:
df_test_results

,ID,customer_id,product_unit_variant_id,week_start,product_id,grade_name,unit_name,product_grade_variant_id,customer_category,customer_status,...,unit_name_UNIT_000,unit_name_UNIT_007,unit_name_UNIT_002,unit_name_UNIT_005,unit_name_UNIT_003,unit_name_UNIT_009,customer_id_encoded,product_id_encoded,prediction_proba,prediction_binary
0,438_278_20250922,438,278,2025-09-22,157,GRADE_01,UNIT_004,202,CUST_CAT_007,CUST_STAT_000,...,0,0,0,0,0,0,31,54,0.281511,0
1,367_179_20250922,367,179,2025-09-22,135,GRADE_01,UNIT_007,177,CUST_CAT_002,CUST_STAT_000,...,0,1,0,0,0,0,5,34,0.466612,0
2,637_130_20250922,637,130,2025-09-22,83,GRADE_01,UNIT_004,104,CUST_CAT_001,CUST_STAT_000,...,0,0,0,0,0,0,89,206,0.047598,0
3,568_62_20250922,568,62,2025-09-22,76,GRADE_01,UNIT_004,96,CUST_CAT_003,CUST_STAT_000,...,0,0,0,0,0,0,76,198,0.567130,1
4,667_168_20250922,667,168,2025-09-22,101,GRADE_01,UNIT_004,130,CUST_CAT_003,CUST_STAT_000,...,0,0,0,0,0,0,102,3,0.295273,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45961,367_6_20250922,367,6,2025-09-22,22,GRADE_01,UNIT_004,32,CUST_CAT_002,CUST_STAT_000,...,0,0,0,0,0,0,5,71,0.595627,1
45962,625_173_20250922,625,173,2025-09-22,82,GRADE_01,UNIT_004,103,CUST_CAT_003,CUST_STAT_002,...,0,0,0,0,0,0,85,205,0.248647,0
45963,713_178_20250922,713,178,2025-09-22,79,GRADE_06,UNIT_004,99,CUST_CAT_003,CUST_STAT_000,...,0,0,0,0,0,0,116,201,0.246068,0
45964,721_337_20250922,721,337,2025-09-22,350,GRADE_01,UNIT_004,356,CUST_CAT_003,CUST_STAT_000,...,0,0,0,0,0,0,122,105,0.213412,0


In [12]:
# features